# 1. Metadata

In [ ]:
# ============================================================
# NOTEBOOK METADATA
# ============================================================

NOTEBOOK_NAME = "02_asset_characterization_momentum"
AUTHOR = "Juan Manuel Giacame"
CREATED = "2026-03-08"
UPDATED = "2026-03-09"
PROJECT = "macro-market-analysis"
STAGE = "Research"
VERSION = "0.1.0"
GIT_HASH = ""  # completar con commit hash si se usa git
EXPERIMENT_ID = f"{NOTEBOOK_NAME}_{CREATED}_{VERSION}"

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
PURPOSE = """
Compute momentum features for each asset in the universe.

This notebook calculates time-series momentum metrics using
rolling return aggregation across multiple horizons.

The resulting features are stored per asset in:

data/features/{asset}/momentum.parquet
"""

# ------------------------------------------------------------
# INPUT DATASETS
# ------------------------------------------------------------
INPUT_DATASETS = [
    "data/processed/prices.parquet"
]

# ------------------------------------------------------------
# OUTPUT DATASETS
# ------------------------------------------------------------
OUTPUT_DATASETS = [
    "data/features/{asset}/momentum.parquet"
]

# ------------------------------------------------------------
# FEATURE FAMILY
# ------------------------------------------------------------
FEATURE_FAMILY = "Momentum"

# ------------------------------------------------------------
# FEATURE DESCRIPTION
# ------------------------------------------------------------
FEATURE_DESCRIPTION = """
Momentum features measure the cumulative return of an asset
over multiple historical horizons.

These features capture trend persistence and are commonly used
in time-series momentum and cross-sectional momentum strategies.
"""

# ------------------------------------------------------------
# ASSETS UNIVERSE
# ------------------------------------------------------------
ASSETS_UNIVERSE = "All assets in data/processed/prices.parquet"

# ------------------------------------------------------------
# DEPENDENCIES
# ------------------------------------------------------------
DEPENDENCIES = ["pandas >= 2.0", "numpy", "plotly >= 5.0", "pathlib"]

# ------------------------------------------------------------
# NOTES
# ------------------------------------------------------------
NOTES = """
Features are computed independently per asset.

Cross-sectional transformations such as:
    - rankings
    - z-scores
    - dispersion
are handled later in:

03_systemic_features.ipynb
"""

# 2. Imports

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

# -----------------------------
# Standard library
# -----------------------------
from pathlib import Path

# -----------------------------
# Data manipulation
# -----------------------------
import pandas as pd  # pandas >= 2.0
import numpy as np   # numpy >= 1.25

# Optional: display settings for research notebooks
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.options.mode.chained_assignment = None  # avoid SettingWithCopyWarning

# -----------------------------
# Parquet engine
# -----------------------------
import pyarrow  # pyarrow >= 12.0

# -----------------------------
# Visualization
# -----------------------------
import plotly.express as px  # plotly >= 5.0
import plotly.graph_objects as go

# -----------------------------
# Optional future imports from src/ once implemented
# -----------------------------
# from src.utils import panel_utils, plotting_utils

# 3. Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# DATA PATHS
# ------------------------------------------------------------
DATA_DIR = Path("../../data")
PROCESSED_DATA_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features"

# Input dataset for momentum calculations
PRICES_PATH = PROCESSED_DATA_DIR / "prices.parquet"

# ------------------------------------------------------------
# FEATURE FAMILY
# ------------------------------------------------------------
FEATURE_FAMILY = "Momentum"

# ------------------------------------------------------------
# FEATURE PARAMETERS
# ------------------------------------------------------------
LOOKBACK_WINDOWS = [21, 63, 126, 252]

# Smoothing windows for velocity/acceleration derivatives (VEL_S, ACC_S)
SMOOTH_WINDOWS = {
    21: 3,
    63: 3,
    126: 5,
    252: 5
}

# ------------------------------------------------------------
# UNIVERSE CONFIGURATION
# ------------------------------------------------------------
# Will be dynamically extracted from df_prices after loading
ASSETS = None
UNIVERSE_SOURCE = "prices_columns"

# ------------------------------------------------------------
# OUTPUT CONFIGURATION
# ------------------------------------------------------------
EXPORT_FORMAT = "parquet"
OVERWRITE_EXISTING = True

# ------------------------------------------------------------
# RESEARCH OPTIONS
# ------------------------------------------------------------
# Sample assets for visualizations and sanity checks
SAMPLE_ASSETS_FOR_PLOTS = [
    "BTC",
    "SPY"
]

# ------------------------------------------------------------
# RANDOM SEED
# ------------------------------------------------------------
RANDOM_SEED = 42

# 4. Data Loading

In [ ]:
# ============================================================
# 4. DATA LOADING
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Detect project root by searching for 'data' directory
# ------------------------------------------------------------
current_path = Path().resolve()

project_root = None
for parent in [current_path] + list(current_path.parents):
    if (parent / "data").exists():
        project_root = parent
        break

if project_root is None:
    raise RuntimeError("Project root with 'data/' directory not found.")

print("Project root detected at:", project_root)

# ------------------------------------------------------------
# Define data paths
# ------------------------------------------------------------
DATA_DIR = project_root / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features"

PRICES_PATH = PROCESSED_DATA_DIR / "prices.parquet"

print("\nLooking for prices dataset:", PRICES_PATH)

# ------------------------------------------------------------
# Verify files exist
# ------------------------------------------------------------
if not PRICES_PATH.exists():
    raise FileNotFoundError(f"Prices dataset not found at {PRICES_PATH}")

print("Prices dataset found ✔")

# ------------------------------------------------------------
# Load prices dataset
# ------------------------------------------------------------
df_prices = pd.read_parquet(PRICES_PATH)
df_prices = df_prices.sort_index()

# Ensure datetime index
if "date" in df_prices.columns:
    df_prices["date"] = pd.to_datetime(df_prices["date"])
    df_prices.set_index("date", inplace=True)

df_prices.sort_index(inplace=True)

print("Prices dataset shape:", df_prices.shape)

# ------------------------------------------------------------
# Infer asset universe
# ------------------------------------------------------------
ASSETS = df_prices.columns.tolist()
print("\nNumber of assets in universe:", len(ASSETS))
print("Sample assets:", ASSETS[:10])

# ------------------------------------------------------------
# Quick inspection
# ------------------------------------------------------------
display(df_prices.head())

# 5. Core Computation
MOMENTUM

Momentum represents the persistence of price movements over time.
Assets exhibiting positive momentum tend to continue outperforming,
while negative momentum assets tend to underperform.

Trend analysis allows us to quantify:

- Direction of movement
- Strength of persistence
- Stability of trends
- Speed of price evolution

Momentum features are fundamental inputs for both
asset allocation and alpha generation models.

In this section we construct statistical measures that
capture trend behavior across multiple horizons.

### 5.1 Multi-Horizon Momentum

Momentum measures the persistence of price trends over time.

Instead of relying on a single lookback window, we compute momentum
across multiple horizons to capture short-, medium-, and long-term trends.

This approach allows us to characterize how trend strength evolves
across different investment horizons.

Momentum is computed using cumulative log returns:

$$ Momentum(t, n) = log(P_t / P_{t-n}) $$

Where:

$ - P_t$ = current price 
$ - P_{t-n}$ = price n periods ago 
$ - n$ = lookback window 

We compute momentum over multiple horizons:

- 21 days  → Short-term momentum (~1 month)
- 63 days  → Medium-term momentum (~3 months)
- 126 days → Intermediate trend (~6 months)
- 252 days → Long-term momentum (~1 year)

Multi-horizon momentum helps identify trend persistence
and regime transitions across assets.

Structural NaNs naturally appear during the warmup period and are preserved.
These observations will later be excluded when defining the research dataset.

In [ ]:
# ============================================================
# 5. CORE COMPUTATIONS
# Feature Block: Momentum Level (MOM_h)
# ============================================================

# ------------------------------------------------------------
# Compute log momentum for each horizon
# ------------------------------------------------------------
momentum_level = {
    f"MOM_{h}": np.log(df_prices / df_prices.shift(h))
    for h in LOOKBACK_WINDOWS
}

# ------------------------------------------------------------
# Concatenate into a single DataFrame
# MultiIndex columns: feature x asset
# ------------------------------------------------------------
momentum_level = pd.concat(momentum_level, axis=1)
momentum_level.columns.names = ["feature", "asset"]

# Ensure consistent column ordering
momentum_level = momentum_level.sort_index(axis=1)

# ------------------------------------------------------------
# Note on NaNs
# ------------------------------------------------------------
# NaNs are expected due to lookback windows (warmup period)

# ------------------------------------------------------------
# Quick inspection
# ------------------------------------------------------------
print("Momentum Level features created:")
print(momentum_level.columns)

print("Dataset shape:", momentum_level.shape)

momentum_level.head()

### 5.2 Momentum Stability

Momentum alone does not indicate whether a trend is reliable.

Momentum Stability measures how consistent momentum remains
through time. Stable momentum suggests persistent trends,
while unstable momentum indicates noisy or fragile price movements.

We estimate momentum stability as the rolling standard deviation
of momentum values.

Lower dispersion implies stronger trend persistence.

Mathematically:

$$ Momentum Stability = std(Momentum_t) over rolling window $$

Where:

- Low values → stable trend
- High values → unstable or reversing trend

This metric helps distinguish between sustainable trends
and short-lived market moves.

In [ ]:
# ==========================================
# 5. CORE COMPUTATIONS
# Feature Block: Momentum Stability (MOM_h_STAB)
# ==========================================

# Dictionary to store stability features
momentum_stab = {}

# Loop over each momentum horizon
for h in LOOKBACK_WINDOWS:
    
    feature_name = f"MOM_{h}_STAB"
    
    # Rolling standard deviation of MOM_h
    stab = momentum_level[f"MOM_{h}"].rolling(h).std()
    
    # Store result
    momentum_stab[feature_name] = stab


# Concatenate into MultiIndex DataFrame
momentum_stab = pd.concat(momentum_stab, axis=1)
momentum_stab.columns.names = ["feature", "asset"]

# Sort columns for consistent ordering
momentum_stab.sort_index(axis=1, inplace=True)


# Quick inspection
print("Momentum Stability features created:")
print(momentum_stab.columns)

momentum_stab.head()

### 5.3 Momentum Z-Score (Normalized Trend Strength)

Raw momentum values are not directly comparable across assets,
as each asset exhibits different volatility and trend magnitude.

To normalize trend strength, momentum is standardized using
a rolling Z-score.

Z-score measures how extreme the current momentum is relative
to its historical distribution.

Mathematically:

$$ Z_t = (MOM_t - μ_t) / σ_t "" $$

Where:

$ - MOM_t $ : momentum at time t 
$ - μ_t $  : rolling mean of momentum 
$ - σ_t $ : rolling standard deviation 

Interpretation:

- Z > 2  → unusually strong trend
- Z ≈ 0  → normal trend conditions
- Z < -2 → unusually weak or bearish trend

This transformation allows comparison of trend strength
across assets and time.

In [ ]:
# ============================================================
# 5. CORE COMPUTATIONS
# Feature Block: Standardized Momentum (MOM_h_Z)
# ============================================================

# Rolling window for z-score normalization (1 year)
Z_WINDOW = 252

momentum_z = {}

# ------------------------------------------------------------
# Compute rolling z-score of momentum
# ------------------------------------------------------------
for h in LOOKBACK_WINDOWS:

    mom = momentum_level[f"MOM_{h}"]

    rolling_mean = mom.rolling(Z_WINDOW).mean()
    rolling_std  = mom.rolling(Z_WINDOW).std()

    momentum_z[f"MOM_{h}_Z"] = (mom - rolling_mean) / rolling_std

# ------------------------------------------------------------
# Concatenate into single DataFrame
# ------------------------------------------------------------
momentum_z = pd.concat(momentum_z, axis=1)
momentum_z.columns.names = ["feature", "asset"]

# Consistent ordering
momentum_z = momentum_z.sort_index(axis=1)

# ------------------------------------------------------------
# NaNs are expected due to rolling windows
# ------------------------------------------------------------

# Quick inspection
print(f"Standardized Momentum features (rolling {Z_WINDOW}d) created:")
print(momentum_z.columns)
print("Dataset shape:", momentum_z.shape)

momentum_z.head()

### 5.4 Momentum Agreement Index

Momentum signals across different horizons may diverge.

The Momentum Agreement Index measures whether short-, medium-,
and long-term momentum signals point in the same direction.

Trend conviction increases when multiple horizons agree.

We define agreement as the average sign of momentum signals:

$$ Agreement = mean(sign(Momentum_i)) $$

Where Momentum_i represents momentum computed
across multiple lookback windows.

Interpretation:

- +1 → Strong bullish agreement
- 0  → Mixed or transition regime
- -1 → Strong bearish agreement

This metric helps detect high-conviction trends
and potential regime shifts.

In [ ]:
# ==========================================
# Feature Block: Momentum Alignment (cross-horizon agreement)
# ==========================================

assets = momentum_level.columns.levels[1]
alignment_horizons = [f"MOM_{h}" for h in LOOKBACK_WINDOWS]

# Dictionary to store alignment per asset
momentum_align_dict = {}

for asset in assets:
    
    # Subset momentum features for the current asset
    df_asset = momentum_level.xs(asset, level="asset", axis=1)
    
    # Compute sign of momentum for each horizon
    momentum_sign = df_asset[alignment_horizons].apply(np.sign)
    
    # Average sign across horizons (agreement index)
    momentum_align_dict[asset] = momentum_sign.mean(axis=1)

# Concatenate results across assets
momentum_align = pd.concat(momentum_align_dict, axis=1)

# Recreate MultiIndex structure
momentum_align.columns = pd.MultiIndex.from_product(
    [["MOM_ALIGN"], assets]
)
momentum_align.columns.names = ["feature", "asset"]

# Sort columns for consistency
momentum_align.sort_index(axis=1, inplace=True)

print("Momentum Alignment feature created:")
print(momentum_align.columns)

## 5.4.1 Momentum Agreement Index (Z-Score Based)

Objective

Measure multi-horizon trend alignment using standardized momentum (Z-scores) 
instead of raw momentum values.

While raw momentum agreement captures directional alignment, it treats weak 
and strong signals equally. Using Z-scores allows us to account for statistical 
significance relative to historical behavior.

---

Economic Intuition

Markets exhibit stronger regime behavior when multiple time horizons 
show statistically significant trend alignment.

If:

- Short-term momentum is positive,
- Medium-term momentum is positive,
- Long-term momentum is positive,

and all are meaningfully different from zero (high |Z|),

then the probability of a persistent bullish regime increases.

The same logic applies for bearish alignment.

---

Construction

For each asset and date:

1. Compute the sign of standardized momentum (Z-score).
2. Average the signs across all horizons.
3. Optionally filter weak signals (|Z| below threshold).

The resulting index lies in:

- +1  → all horizons significantly bullish
-  0  → mixed or weak signals
- -1  → all horizons significantly bearish

---

Interpretation

This feature represents a:

Multi-Scale Significant Trend Alignment Score

It can later be used for:
- Regime detection
- Signal filtering
- Conditional allocation rules

In [ ]:
# ==========================================
# Momentum Agreement Index (Z-Score Filtered)
# ==========================================

# Available assets
assets = momentum_z.columns.levels[1]

# Z-score momentum features
alignment_horizons = momentum_z.columns.levels[0]

# Z-score significance threshold
Z_THRESHOLD = 0.5

# Dictionary to store alignment per asset
momentum_align_dict = {}

for asset in assets:
    
    # Select z-scored momentum for the asset
    df_asset = momentum_z.xs(asset, level="asset", axis=1)
    
    # Extract Z-score horizons
    z_values = df_asset[alignment_horizons]
    
    # Filter weak signals while preserving NaNs
    z_filtered = z_values.mask(np.abs(z_values) < Z_THRESHOLD)
    
    # Compute sign of filtered signals
    z_sign = np.sign(z_filtered)
    
    # Average sign across horizons
    momentum_align_dict[asset] = z_sign.mean(axis=1)

# ------------------------------------------
# Build final DataFrame
# ------------------------------------------

momentum_align_z = pd.concat(momentum_align_dict, axis=1)

momentum_align_z.columns = pd.MultiIndex.from_product(
    [["MOM_ALIGN_Z"], assets]
)

momentum_align_z.columns.names = ["feature", "asset"]

momentum_align_z.sort_index(axis=1, inplace=True)

print("Filtered Momentum Agreement Index created successfully:")
print(momentum_align_z.head())

In [ ]:
# ------------------------------------------
# Structure checks
# ------------------------------------------

print("Shape:")
print(momentum_align.shape)

print("\nColumn levels:")
print(momentum_align.columns.names)

print("\nFeature names:")
print(momentum_align.columns.levels[0])

print("\nAssets:")
print(momentum_align.columns.levels[1][:10])

print("\nIndex sample:")
print(momentum_align.index[:5])

print("\nHead:")
print(momentum_align.tail())

In [ ]:
momentum_align.stack().hist(bins=50)

In [ ]:
momentum_align.describe()

## 5.5 Cross-Sectional Momentum Rank

Objective

Measure relative strength across assets at each point in time.

While time-series momentum tells us whether an asset is trending,
cross-sectional momentum tells us whether an asset is outperforming 
its peers.

This is essential for asset allocation and rotation strategies.

---

Economic Intuition

At each date:

- Rank assets by momentum within the universe.
- Assign a percentile score between 0 and 1.

Interpretation:

1.0 → strongest asset in the universe  
0.0 → weakest asset  
0.5 → neutral relative position  

This captures relative leadership and capital rotation dynamics.

---

Construction

For each momentum horizon:

1. At each date, rank assets cross-sectionally.
2. Convert rank into percentile form.
3. Preserve multi-horizon structure.

---

Use Cases

- Long strongest / short weakest
- Risk-parity tilts
- Allocation weighting schemes
- ML ranking features

In [ ]:
# ==========================================
# Cross-Sectional Momentum Rank
# ==========================================

# Compute cross-sectional percentile rank per horizon

momentum_cs = (
    momentum_level
        .T
        .groupby(level="feature")
        .rank(pct=True)
        .T
)

# Rename features to indicate cross-sectional ranking
momentum_cs.columns = pd.MultiIndex.from_tuples(
    [(f"{feat}_CS", asset) for feat, asset in momentum_cs.columns],
    names=momentum_cs.columns.names
)

# Ensure consistent column ordering
momentum_cs.sort_index(axis=1, inplace=True)

print("Cross-sectional momentum rank created successfully:")
momentum_cs.tail()

In [ ]:
momentum_cs.describe()

## 5.6 Cross-Sectional Momentum Z-Score

Objective

Standardize momentum relative to the universe at each date.

Instead of ranking assets, compute how many standard deviations 
an asset's momentum is from the cross-sectional mean.

---
Economic Intuition

If all assets are rising, raw momentum is less informative.

Cross-sectional Z-score measures:

"How exceptional is this asset relative to peers right now?"

Positive Z → outperforming universe  
Negative Z → underperforming universe  

---
 Construction

For each date and horizon:

1. Compute cross-sectional mean.
2. Compute cross-sectional standard deviation.
3. Standardize each asset’s momentum.

In [ ]:
# ==========================================
# Cross-Sectional Momentum Z-Score
# ==========================================

momentum_cs_z = (
    momentum_z
        .T
        .groupby(level="feature")
        .transform(lambda x: (x - x.mean()) / x.std())
        .T
)

# Rename features
momentum_cs_z.columns = pd.MultiIndex.from_tuples(
    [(feat.replace("_Z", "_CS_Z"), asset) for feat, asset in momentum_cs_z.columns],
    names=momentum_cs_z.columns.names
)

# Ensure consistent ordering
momentum_cs_z.sort_index(axis=1, inplace=True)

momentum_cs_z.tail()

## 5.7 Momentum Dinamics


In [ ]:
# ==========================================
# Momentum Velocity
# ==========================================

# Compute velocity as first difference of momentum_level
momentum_vel = momentum_level.diff()

# Rename columns to keep MultiIndex convention
momentum_vel.columns = pd.MultiIndex.from_tuples(
    [(f"{feat}_VEL", asset) for feat, asset in momentum_vel.columns],
    names=momentum_vel.columns.names
)

# Quick inspection
print("Momentum Velocity features created:")
print(momentum_vel.columns)
momentum_vel.head()


# ==========================================
# Momentum Acceleration
# ==========================================

# Compute acceleration as difference of velocity
momentum_acc = momentum_vel.diff()

# Rename columns carefully: replace _VEL with _ACC
momentum_acc.columns = pd.MultiIndex.from_tuples(
    [(feat.replace("_VEL", "_ACC"), asset) for feat, asset in momentum_acc.columns],
    names=momentum_acc.columns.names
)
#
# NANs are espected due to Warm up

# Quick inspection
print("Momentum Acceleration features created:")
print(momentum_acc.columns)
momentum_acc.head()

In [ ]:
# ==========================================
# 5. CORE COMPUTATIONS
# Feature Block: Dynamics (variable smoothing)
# ==========================================

# Dictionaries to store smoothed features
momentum_vel_s = {}
momentum_acc_s = {}

# Loop over each momentum horizon
for h in LOOKBACK_WINDOWS:
    
    # Retrieve smoothing window from configuration
    smooth_w = SMOOTH_WINDOWS[h]
    
    # Feature names
    vel_s_name = f"MOM_{h}_VEL_S"
    acc_s_name = f"MOM_{h}_ACC_S"
    
    # Retrieve raw velocity and acceleration
    vel_raw = momentum_vel[f"MOM_{h}_VEL"]
    acc_raw = momentum_acc[f"MOM_{h}_ACC"]
    
    # Apply variable rolling smoothing
    vel_s = vel_raw.rolling(smooth_w).mean()
    acc_s = acc_raw.rolling(smooth_w).mean()
    
    # Store results
    momentum_vel_s[vel_s_name] = vel_s
    momentum_acc_s[acc_s_name] = acc_s


# Concatenate into MultiIndex DataFrames
momentum_vel_s = pd.concat(momentum_vel_s, axis=1)
momentum_acc_s = pd.concat(momentum_acc_s, axis=1)

# Clean and organize columns
for df in [momentum_vel_s, momentum_acc_s]:
    
    # Set MultiIndex column names
    df.columns.names = ["feature", "asset"]
    
    # Sort columns for readability
    df.sort_index(axis=1, inplace=True)
    



# Quick inspection
print("Momentum Dynamics (variable smoothing) features created:")
print(momentum_vel_s.columns)
print(momentum_acc_s.columns)

momentum_vel_s.head()

In [ ]:
# ==========================================
# Momentum Dynamics Consistency Check con %
# ==========================================

consistency_check = {}

for h in LOOKBACK_WINDOWS:
    
    # Raw vs smoothed
    vel_raw  = momentum_vel[f"MOM_{h}_VEL"]
    vel_s    = momentum_vel_s[f"MOM_{h}_VEL_S"]
    
    acc_raw  = momentum_acc[f"MOM_{h}_ACC"]
    acc_s    = momentum_acc_s[f"MOM_{h}_ACC_S"]
    
    # Alinear índices y columnas
    vel_raw_aligned, vel_s_aligned = vel_raw.align(vel_s, join="inner")
    acc_raw_aligned, acc_s_aligned = acc_raw.align(acc_s, join="inner")
    
    # Correlation por asset
    corr_vel = vel_raw_aligned.corrwith(vel_s_aligned)
    corr_acc = acc_raw_aligned.corrwith(acc_s_aligned)
    
    # Varianza de raw y smoothed
    var_vel_raw = vel_raw_aligned.var()
    var_vel_s   = vel_s_aligned.var()
    var_acc_raw = acc_raw_aligned.var()
    var_acc_s   = acc_s_aligned.var()
    
    # % Reducción de varianza
    vel_var_reduction_pct = ((var_vel_raw - var_vel_s) / var_vel_raw * 100).mean()
    acc_var_reduction_pct = ((var_acc_raw - var_acc_s) / var_acc_raw * 100).mean()
    
    # Sign flips
    sign_flips_vel = (np.sign(vel_raw_aligned) != np.sign(vel_s_aligned))
    sign_flips_acc = (np.sign(acc_raw_aligned) != np.sign(acc_s_aligned))
    
    # % de cambio de signo
    vel_sign_flips_pct = sign_flips_vel.sum().sum() / np.prod(sign_flips_vel.shape) * 100
    acc_sign_flips_pct = sign_flips_acc.sum().sum() / np.prod(sign_flips_acc.shape) * 100
    
    # Guardar resultados
    consistency_check[h] = {
        "vel_corr_mean": corr_vel.mean(),
        "vel_var_reduction_pct": vel_var_reduction_pct,
        "vel_sign_flips_pct": vel_sign_flips_pct,
        "acc_corr_mean": corr_acc.mean(),
        "acc_var_reduction_pct": acc_var_reduction_pct,
        "acc_sign_flips_pct": acc_sign_flips_pct
    }

# Convertir a DataFrame para inspección
consistency_df = pd.DataFrame(consistency_check).T
consistency_df

## 5.9 Momentum Strength Index (Weighted & Smoothed)

Individual momentum horizons capture trend dynamics at
different temporal scales. However, analyzing them
separately does not provide a unified structural view
of market strength.

To aggregate cross-horizon information, we define the
Momentum Strength Index (MSI) as a weighted combination
of normalized momentum signals.

Let $ Z_t(h) $ denote the Z-scored momentum for horizon h.

The weighted MSI is defined as:

$$ MSI_t = Σ_h ( w_h * Z_t(h) ) $$

where weights satisfy:

$$ Σ_h w_h = 1 $$

The weighting scheme reflects structural importance
across horizons.

In our baseline specification:

w_21  = 0.1  
w_63  = 0.2  
w_126 = 0.3  
w_252 = 0.4  

Longer horizons receive higher weights to emphasize
structural regime components over tactical noise.

Since MSI aggregates multiple horizons, it is further
smoothed to enhance regime stability:

$$ M̃SI_t = (1 / k) * Σ_{i=0}^{k-1} MSI_{t-i} $$

with k = 5.

This produces a structurally stable, cross-horizon
trend-strength indicator suitable for regime detection
and allocation modeling.

In [ ]:
# ==========================================
# Momentum Strength Index (MSI)
# ==========================================

weights = {
    "MOM_21_Z": 0.1,
    "MOM_63_Z": 0.2,
    "MOM_126_Z": 0.3,
    "MOM_252_Z": 0.4,
}

msi_components = []

for feat, w in weights.items():
    
    # Select Z-scored momentum for the horizon
    df_feat = momentum_z.xs(feat, level="feature", axis=1)
    
    # Apply weight
    df_weighted = df_feat * w
    
    msi_components.append(df_weighted)

# Aggregate weighted horizons
msi_raw = pd.concat(msi_components).groupby(level=0).sum()

# Restore MultiIndex structure
msi_raw.columns = pd.MultiIndex.from_product(
    [["MSI"], msi_raw.columns],
    names=momentum_z.columns.names
)

# ------------------------------------------
# Smooth the index
# ------------------------------------------

msi_smooth = msi_raw.rolling(window=5).mean()

msi_smooth.columns = pd.MultiIndex.from_product(
    [["MSI_S"], msi_smooth.columns.get_level_values("asset")],
    names=msi_raw.columns.names
)

print("Momentum Strength Index created successfully:")
msi_smooth.tail()

In [ ]:
msi_smooth.stack().hist(bins=50)

## 5.9.1 MSI Velocity (Smoothed)

To capture structural changes in aggregated trend strength,
we compute the first derivative of the smoothed MSI.

MSI velocity is defined as:

$$ MSI_{Vel_t} = M̃SI_t - M̃SI_{t-1} $$

This measures the rate of change in structural momentum
across horizons.

Because MSI is already smoothed, velocity reflects
structural acceleration rather than tactical noise.

To ensure stability, MSI velocity is additionally
smoothed using a 5-day rolling mean:

$$ M̃SI_{Vel_t} = (1 / 5) * Σ_{i=0}^{4} MSI_{Vel_{t-i}} $$

This produces a stable structural force indicator,
useful for identifying regime transitions.

In [ ]:
# MSI velocity
msi_velocity = msi_smooth.diff()

# Smoothed velocity
msi_velocity_smooth = (
    msi_velocity
        .rolling(window=5)
        .mean()
)

# Asignar MultiIndex consistente
msi_velocity_smooth.columns = pd.MultiIndex.from_arrays(
    [
        ["MSI_VEL_S"] * len(msi_velocity_smooth.columns),
        msi_velocity_smooth.columns.get_level_values("asset")
    ],
    names=msi_velocity_smooth.columns.names
)

msi_velocity_smooth.columns

## 5.9.2 MSI Acceleration (Smoothed)

To detect structural inflection points, we compute
the second derivative of the smoothed MSI.

MSI acceleration is defined as:

$$ MSI_{Acc_t} = MSI_{Vel_t} - MSI_{Vel_{t-1}} $$

Equivalently:

$$ MSI_{Acc_t} = Δ² M̃SI_t $$

Acceleration captures changes in structural momentum force,
identifying reinforcement or exhaustion of dominant regimes.

Given the noise amplification inherent in second derivatives,
MSI acceleration is smoothed using a 5-day rolling mean:

$$ M̃SI_{Acc_t} = (1 / 5) * Σ_{i=0}^{4} MSI_{Acc_{t-i}} $$

The resulting signal provides a stable structural
inflection indicator suitable for regime-state modeling.

In [ ]:
msi_acceleration = msi_velocity.diff()

msi_acceleration_smooth = (
    msi_acceleration
        .rolling(window=5)
        .mean()
)

msi_acceleration_smooth.columns = pd.MultiIndex.from_product(
    [["MSI_ACC_S"], msi_acceleration.columns.get_level_values("asset")],
    names=msi_acceleration.columns.names
)

msi_acceleration_smooth.columns






In [ ]:
msi_acceleration = msi_velocity.diff()

msi_acceleration_smooth = (
    msi_acceleration
        .rolling(window=5)
        .mean()
)

msi_acceleration_smooth.columns = pd.MultiIndex.from_arrays(
    [
        ["MSI_ACC_S"] * len(msi_acceleration_smooth.columns),
        msi_acceleration_smooth.columns.get_level_values("asset")
    ],
    names=msi_acceleration_smooth.columns.names
)

msi_acceleration_smooth.columns

In [ ]:
momentum_structural = pd.concat([
    msi_raw,
    msi_smooth,
    msi_velocity_smooth,
    msi_acceleration_smooth
], axis=1).sort_index(axis=1)

momentum_structural.columns

In [ ]:
asset = "BTC"  # ajustar si es necesario

msi_r = msi_raw.xs("MSI", level="feature", axis=1)[asset]
msi_s = msi_smooth.xs("MSI_S", level="feature", axis=1)[asset]

# Align
df = pd.concat([msi_r, msi_s], axis=1).dropna()
df.columns = ["raw", "smooth"]

# Correlation
corr = df["raw"].corr(df["smooth"])

# Variance reduction
std_raw = df["raw"].std()
std_smooth = df["smooth"].std()
var_reduction = 1 - (std_smooth / std_raw)

# Sign flips
sign_flips_raw = (df["raw"].diff().apply(np.sign).diff() != 0).sum()
sign_flips_smooth = (df["smooth"].diff().apply(np.sign).diff() != 0).sum()
flip_reduction = 1 - (sign_flips_smooth / sign_flips_raw)

print("Correlation:", round(corr,4))
print("Std raw:", std_raw)
print("Std smooth:", std_smooth)
print("Variance reduction (%):", round(var_reduction*100,2))
print("Sign flips raw:", sign_flips_raw)
print("Sign flips smooth:", sign_flips_smooth)
print("Flip reduction (%):", round(flip_reduction*100,2))

In [ ]:
vel_r = msi_velocity.xs("MSI_S", level="feature", axis=1)[asset]
vel_s = msi_velocity_smooth.xs("MSI_VEL_S", level="feature", axis=1)[asset]

df_vel = pd.concat([vel_r, vel_s], axis=1).dropna()
df_vel.columns = ["raw", "smooth"]

corr_v = df_vel["raw"].corr(df_vel["smooth"])
std_raw_v = df_vel["raw"].std()
std_smooth_v = df_vel["smooth"].std()
var_red_v = 1 - (std_smooth_v / std_raw_v)

sign_flips_raw_v = (df_vel["raw"].diff().apply(np.sign).diff() != 0).sum()
sign_flips_smooth_v = (df_vel["smooth"].diff().apply(np.sign).diff() != 0).sum()
flip_red_v = 1 - (sign_flips_smooth_v / sign_flips_raw_v)

print("Velocity Correlation:", round(corr_v,4))
print("Velocity Variance reduction (%):", round(var_red_v*100,2))
print("Velocity Flip reduction (%):", round(flip_red_v*100,2))

In [ ]:
sorted(set(momentum_structural.columns.get_level_values("feature")))

## 5.10 Concatenation 

In [ ]:
# ==========================================
# 5. CORE COMPUTATIONS
# Concatenación final de todas las features de Momentum
# ==========================================

# Lista de DataFrames por bloques ya calculados
momentum_blocks = [
    momentum_level,       # MOM_h
    momentum_z,           # MOM_h_Z
    momentum_vel,         # MOM_h_VEL
    momentum_vel_s,       # MOM_h_VEL_S
    momentum_acc,         # MOM_h_ACC
    momentum_acc_s,       # MOM_h_ACC_S
    momentum_stab,        # MOM_h_STAB
    momentum_align,       # MOM_ALIGN
    momentum_align_z,     # MOM_ALIGN_Z
    momentum_structural,  # MSI_RAW, MSI_S, MSI_VEL_S, MSI_ACC_S
    momentum_cs,          # MOM_h_CS
    momentum_cs_z         # MOM_h_CS_Z
]

# ------------------------------------------
# Concatenación
# ------------------------------------------

momentum_final = pd.concat(momentum_blocks, axis=1)

# ------------------------------------------
# Validación estructura MultiIndex
# ------------------------------------------

assert momentum_final.columns.nlevels == 2, "Columns must be a 2-level MultiIndex."

# Asegurar nombres consistentes
momentum_final.columns.names = ["feature", "asset"]

# ------------------------------------------
# Detectar columnas duplicadas
# ------------------------------------------

duplicated_cols = momentum_final.columns.duplicated()

if duplicated_cols.any():
    
    dup_list = momentum_final.columns[duplicated_cols]
    
    raise ValueError(
        f"Duplicate feature columns detected:\n{dup_list}"
    )

# ------------------------------------------
# Orden consistente (feature → asset)
# ------------------------------------------

momentum_final.sort_index(axis=1, inplace=True)

# ------------------------------------------
# Quick inspection
# ------------------------------------------

print("Momentum final concatenado correctamente.")
print()

print("Column levels:")
print(momentum_final.columns.names)

print()
print("Total features:", momentum_final.columns.get_level_values("feature").nunique())
print("Total assets:", momentum_final.columns.get_level_values("asset").nunique())

print()
print("Preview columns:")
print(momentum_final.columns[:10])

momentum_final.head()

# 6. Diagnosis/Validation

## 6.1 Dataset integrity checks

In [ ]:
# ==========================================
# 6. DIAGNOSTICS / VALIDATION
# ==========================================

# This block validates that the final dataset has been constructed correctly.
# We verify structural integrity, dimensional consistency, and basic data quality.

print("===== DATASET DIMENSIONS =====")

# Validate overall dataset size
# Rows correspond to time observations (dates)
# Columns correspond to feature × asset combinations

print("Rows (dates):", momentum_final.shape[0])
print("Columns (feature × asset):", momentum_final.shape[1])

print()

print("===== MULTIINDEX STRUCTURE =====")

# Validate that the column structure follows the expected MultiIndex format:
# Level 0 → feature
# Level 1 → asset

print("Column levels:", momentum_final.columns.names)

# Count number of unique features and assets
n_features = momentum_final.columns.get_level_values("feature").nunique()
n_assets = momentum_final.columns.get_level_values("asset").nunique()

print("Number of features:", n_features)
print("Number of assets:", n_assets)

print()

print("===== FEATURE LIST =====")

# Inspect the list of computed features
# This helps confirm that all expected feature families were generated

print(momentum_final.columns.get_level_values("feature").unique())

print()

print("===== MISSING DATA CHECK =====")

# Rolling computations and differencing operations naturally create NaN values
# at the beginning of the sample. Here we verify that missing data remains within
# expected levels and no unexpected gaps exist.

nan_counts = momentum_final.isna().sum().sum()
nan_ratio = momentum_final.isna().mean().mean()

print("Total NaN values:", nan_counts)
print("Average NaN ratio:", round(nan_ratio, 4))

print()

print("===== BASIC STATISTICS =====")

# Inspect summary statistics to confirm that signals are within reasonable ranges
# and no numerical instabilities or extreme outliers appear.

print(momentum_final.describe().T.head())

## 6.2 Signal Diagnostics

In [ ]:
# ==========================================
# 6.2 SIGNAL DIAGNOSTICS
# Cross-Sectional Signal Dispersion
# ==========================================

print("===== CROSS-SECTIONAL SIGNAL DISPERSION =====")

# Transpose to have features as rows and assets as columns
momentum_t = momentum_final.T  # now: feature x (date, asset)

# Compute standard deviation across assets for each feature and date
cs_dispersion = momentum_t.groupby(level="feature").std().T  # back to dates x features

# Average dispersion over time for each feature
avg_dispersion = cs_dispersion.mean().sort_values(ascending=False)

print("Average cross-sectional dispersion per feature:")
print(avg_dispersion)

print("\nTop 10 most dispersed signals:")
print(avg_dispersion.head(10))

print("\nBottom 10 least dispersed signals:")
print(avg_dispersion.tail(10))

In [ ]:
# ==========================================
# 6.2 CROSS-SECTIONAL DIAGNOSTICS (CORRECTED)
# ==========================================

# Transpose to have features/assets as rows, dates as columns
df_T = momentum_final.T  # (feature × asset, dates)

# ===============================
# Cross-Sectional Rank Spread
# ===============================
# Rank assets per feature per date (percentile)
cs_rank = df_T.groupby(level=0).apply(lambda df_feat: df_feat.rank(axis=1, pct=True))

# Transpose back to original orientation
cs_rank_T = cs_rank.T

# Compute spread: max - min rank per date, then average over dates per feature
cs_rank_spread = cs_rank_T.apply(lambda row: row.max() - row.min(), axis=1)
# Since row names are (feature, asset), group by feature
cs_rank_spread_by_feature = cs_rank_spread.groupby(level=0).mean()

print("\n===== CROSS-SECTIONAL RANK SPREAD =====")
print(cs_rank_spread_by_feature.sort_values(ascending=False).head(10))  # top 10 most dispersed
print(cs_rank_spread_by_feature.sort_values(ascending=True).head(10))   # bottom 10 least dispersed

# ===============================
# Cross-Sectional Z-Score Dispersion
# ===============================
# Compute standard deviation across assets per date, then average over time
cs_z_dispersion_by_feature = df_T.groupby(level=0).apply(lambda df_feat: df_feat.std(axis=1).mean())

print("\n===== CROSS-SECTIONAL Z-SCORE DISPERSION =====")
print(cs_z_dispersion_by_feature.sort_values(ascending=False).head(10))  # top 10 most dispersed
print(cs_z_dispersion_by_feature.sort_values(ascending=True).head(10))   # bottom 10 least dispersed

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ===============================
# Prepare Data
# ===============================
# Top 10 features by Z-score dispersion
top_z_disp = cs_z_dispersion_by_feature.sort_values(ascending=False).head(10)
bottom_z_disp = cs_z_dispersion_by_feature.sort_values(ascending=True).head(10)

# Top 10 features by rank spread
top_rank_disp = cs_rank_spread_by_feature.sort_values(ascending=False).head(10)
bottom_rank_disp = cs_rank_spread_by_feature.sort_values(ascending=True).head(10)

# ===============================
# Create subplot figure
# ===============================
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Top 10 Z-Score Dispersion",
        "Bottom 10 Z-Score Dispersion",
        "Top 10 Rank Spread",
        "Bottom 10 Rank Spread"
    ),
    vertical_spacing=0.2,
    horizontal_spacing=0.15
)

# Top Z-Score Dispersion
fig.add_trace(go.Bar(x=top_z_disp.index, y=top_z_disp.values, marker_color='crimson'), row=1, col=1)

# Bottom Z-Score Dispersion
fig.add_trace(go.Bar(x=bottom_z_disp.index, y=bottom_z_disp.values, marker_color='darkgreen'), row=1, col=2)

# Top Rank Spread
fig.add_trace(go.Bar(x=top_rank_disp.index, y=top_rank_disp.values, marker_color='orange'), row=2, col=1)

# Bottom Rank Spread
fig.add_trace(go.Bar(x=bottom_rank_disp.index, y=bottom_rank_disp.values, marker_color='blue'), row=2, col=2)

# ===============================
# Layout
# ===============================
fig.update_layout(
    height=800,
    width=1000,
    title_text="Cross-Sectional Momentum Diagnostics",
    showlegend=False
)

fig.update_xaxes(tickangle=-45)

fig.show()

In [ ]:
import plotly.express as px
import pandas as pd

# Conceptual metrics: usar tus métricas reales si las tenés
data = pd.DataFrame({
    "Feature": [
        "MOM_21_CS_Z", "MOM_252_CS_Z", "MOM_126_CS_Z", "MOM_63_CS_Z",
        "MOM_252_Z", "MOM_126_Z", "MOM_63_Z", "MOM_21_Z",
        "MSI", "MSI_S", "MOM_ALIGN_Z", "MOM_ALIGN",
        "MOM_252_ACC_S", "MOM_126_VEL_S", "MSI_ACC_S"
    ],
    "Feature Type": [
        "Z-Score CS", "Z-Score CS", "Z-Score CS", "Z-Score CS",
        "Z-Score TS", "Z-Score TS", "Z-Score TS", "Z-Score TS",
        "Structural", "Structural", "Alignment Z", "Alignment Raw",
        "Smoothed Acceleration", "Smoothed Velocity", "Structural Acceleration"
    ],
    "CS Dispersion": [
        1.0, 1.0, 1.0, 1.0,
        0.983, 0.903, 0.809, 0.724,
        0.672, 0.664, 0.636, 0.513,
        0.005, 0.009, 0.014
    ],
    "CS Rank Spread": [
        0.95, 0.92, 0.88, 0.85,
        0.81, 0.75, 0.70, 0.65,
        0.60, 0.58, 0.55, 0.50,
        0.02, 0.01, 0.015
    ]
})

# Bubble plot: dispersion (y), rank spread (x), color = type
fig = px.scatter(
    data,
    x="CS Rank Spread",
    y="CS Dispersion",
    color="Feature Type",
    size="CS Dispersion",
    hover_name="Feature",
    title="Momentum Feature Diagnostics: Dispersion vs Rank Spread",
    labels={
        "CS Rank Spread": "Cross-Sectional Rank Spread",
        "CS Dispersion": "Cross-Sectional Dispersion"
    },
    height=500
)

fig.update_layout(
    xaxis_range=[0, 1],
    yaxis_range=[0, 1.05],
    legend_title="Feature Type"
)

fig.show()

In [ ]:
import plotly.express as px
import pandas as pd

# Creamos un DataFrame conceptual de ejemplo
# Feature Type: raw, z-scored, velocity, acceleration
# Avg CS Dispersion: promedio de dispersión cross-sectional (0-1)
data = pd.DataFrame({
    "Feature": [
        "MOM_21_CS_Z", "MOM_252_CS_Z", "MOM_126_CS_Z", "MOM_63_CS_Z",
        "MOM_252_Z", "MOM_126_Z", "MOM_63_Z", "MOM_21_Z",
        "MSI", "MSI_S", "MOM_ALIGN_Z", "MOM_ALIGN",
        "MOM_252_ACC_S", "MOM_126_VEL_S", "MSI_ACC_S"
    ],
    "Feature Type": [
        "Z-Score CS", "Z-Score CS", "Z-Score CS", "Z-Score CS",
        "Z-Score TS", "Z-Score TS", "Z-Score TS", "Z-Score TS",
        "Structural", "Structural", "Alignment Z", "Alignment Raw",
        "Smoothed Acceleration", "Smoothed Velocity", "Structural Acceleration"
    ],
    "Avg Cross-Sectional Dispersion": [
        1.0, 1.0, 1.0, 1.0,
        0.98, 0.90, 0.80, 0.72,
        0.67, 0.66, 0.63, 0.51,
        0.005, 0.009, 0.014
    ]
})

# Diagrama de barras interactivo
fig = px.bar(
    data, 
    x="Feature", 
    y="Avg Cross-Sectional Dispersion",
    color="Feature Type",
    text="Avg Cross-Sectional Dispersion",
    title="Momentum Features: Cross-Sectional Dispersion by Type",
    labels={"Avg Cross-Sectional Dispersion": "Average CS Dispersion"}
)

fig.update_layout(
    xaxis_tickangle=-45,
    yaxis_range=[0, 1.05],
    showlegend=True,
    height=500
)

fig.show()

In [ ]:
import plotly.express as px
import pandas as pd

# Example data, reemplazar con métricas reales calculadas
data = pd.DataFrame({
    "Feature": [
        "MOM_21_CS_Z", "MOM_252_CS_Z", "MOM_126_CS_Z", "MOM_63_CS_Z",
        "MOM_252_Z", "MOM_126_Z", "MOM_63_Z", "MOM_21_Z",
        "MSI", "MSI_S", "MOM_ALIGN_Z", "MOM_ALIGN",
        "MOM_252_ACC_S", "MOM_126_VEL_S", "MSI_ACC_S"
    ],
    "Feature Type": [
        "Z-Score CS", "Z-Score CS", "Z-Score CS", "Z-Score CS",
        "Z-Score TS", "Z-Score TS", "Z-Score TS", "Z-Score TS",
        "Structural", "Structural", "Alignment Z", "Alignment Raw",
        "Smoothed Acceleration", "Smoothed Velocity", "Structural Acceleration"
    ],
    "CS Dispersion": [
        1.0, 1.0, 1.0, 1.0,
        0.983, 0.903, 0.809, 0.724,
        0.672, 0.664, 0.636, 0.513,
        0.005, 0.009, 0.014
    ],
    "CS Rank Spread": [
        0.95, 0.92, 0.88, 0.85,
        0.81, 0.75, 0.70, 0.65,
        0.60, 0.58, 0.55, 0.50,
        0.02, 0.01, 0.015
    ]
})

# Plotly scatter with filter by Feature Type
fig = px.scatter(
    data,
    x="CS Rank Spread",
    y="CS Dispersion",
    color="Feature Type",
    size="CS Dispersion",
    hover_name="Feature",
    title="Momentum Feature Diagnostics: Dispersion vs Rank Spread",
    labels={
        "CS Rank Spread": "Cross-Sectional Rank Spread",
        "CS Dispersion": "Cross-Sectional Dispersion"
    },
    height=600
)

# Add dropdown to filter by Feature Type
feature_types = data["Feature Type"].unique()
buttons = [
    dict(
        label=ftype,
        method="update",
        args=[{"visible": data["Feature Type"] == ftype},
              {"title": f"Feature Diagnostics: {ftype}"}]
    )
    for ftype in feature_types
]

# Add "All" button
buttons.insert(0, dict(
    label="All",
    method="update",
    args=[{"visible": [True]*len(data)},
          {"title": "Momentum Feature Diagnostics: All Features"}]
))

fig.update_layout(
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            x=1.15,
            y=1,
            xanchor="right",
            yanchor="top"
        )
    ],
    xaxis_range=[0, 1],
    yaxis_range=[0, 1.05],
    legend_title="Feature Type"
)

fig.show()

# 7. Visualization

In [ ]:
# ============================================================
# SECCIÓN 7: VISUALIZATION / SANITY CHECKS (Plotly robusto)
# ============================================================

import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

# Bloques de features
feature_blocks = ["MOM_", "MOM_CS", "MOM_CS_Z", "MOM_ALIGN", "MOM_ALIGN_Z", "MSI_"]

# ============================================================
# 1. Histogramas por bloque
# ============================================================
for block in feature_blocks:
    # Seleccionamos columnas que empiezan con el bloque
    cols = [c for c in momentum_final.columns.get_level_values("feature") if c.startswith(block)]
    if not cols:
        print(f"No hay columnas para el bloque {block}")
        continue
    
    # Tomamos todas las assets para esas features usando IndexSlice
    df_block = momentum_final.loc[:, pd.IndexSlice[cols, :]]
    
    # Flatten para Plotly (fecha, feature, asset, value)
    df_hist = df_block.stack(level="asset").reset_index()
    df_hist.columns = list(df_hist.columns[:-1]) + ["value"]  # columna de valores como "value"
    
    fig = px.histogram(df_hist, x="value", nbins=50, color="asset",
                       title=f"Distribución de valores del bloque {block}", marginal="box")
    fig.show()

# ============================================================
# 2. Boxplots por asset para un ejemplo de bloque
# ============================================================
example_blocks = ["MOM_252", "MSI_S", "MOM_ALIGN_Z"]
for feat in example_blocks:
    cols = [c for c in momentum_final.columns.get_level_values("feature") if c == feat]
    if not cols:
        continue
    
    df_feat = momentum_final.loc[:, pd.IndexSlice[cols, :]].stack(level="asset").reset_index()
    df_feat.columns = list(df_feat.columns[:-1]) + ["value"]
    
    fig = px.box(df_feat, x="asset", y="value", points="all",
                 title=f"Boxplot de {feat} por asset")
    fig.show()

# ============================================================
# 3. Línea temporal de ejemplo por asset
# ============================================================
example_assets = ["BTC", "QQQ", "SPY"]
for feat in example_blocks:
    fig = go.Figure()
    for asset in example_assets:
        if (feat, asset) in momentum_final.columns:
            fig.add_trace(go.Scatter(x=momentum_final.index,
                                     y=momentum_final[(feat, asset)],
                                     mode="lines",
                                     name=asset))
    fig.update_layout(title=f"Línea temporal de {feat}",
                      xaxis_title="Fecha", yaxis_title="Valor")
    fig.show()

# ============================================================
# 4. Comparación Dynamics raw vs smooth
# ============================================================
dynamics_horizons = [21, 63, 126, 252]
for h in dynamics_horizons:
    vel_col = f"MOM_{h}_VEL"
    vel_s_col = f"MOM_{h}_VEL_S"
    
    fig = go.Figure()
    for asset in example_assets:
        if (vel_col, asset) in momentum_final.columns:
            fig.add_trace(go.Scatter(x=momentum_final.index,
                                     y=momentum_final[(vel_col, asset)],
                                     mode="lines", name=f"VEL {asset}", line=dict(dash="dot")))
        if (vel_s_col, asset) in momentum_final.columns:
            fig.add_trace(go.Scatter(x=momentum_final.index,
                                     y=momentum_final[(vel_s_col, asset)],
                                     mode="lines", name=f"VEL_S {asset}"))
    fig.update_layout(title=f"Momentum VEL vs VEL_S horizonte {h}",
                      xaxis_title="Fecha", yaxis_title="Valor")
    fig.show()

# ============================================================
# 5. Cross-sectional check MOM_CS / MOM_CS_Z
# ============================================================
cs_blocks = ["MOM_CS", "MOM_CS_Z"]
for block in cs_blocks:
    cols = [c for c in momentum_final.columns.get_level_values("feature") if c.startswith(block)]
    if not cols:
        continue
    
    # snapshot final
    df_snapshot = momentum_final.loc[momentum_final.index[-1], pd.IndexSlice[cols, :]]
    df_snapshot = df_snapshot.reset_index()
    df_snapshot.columns = list(df_snapshot.columns[:-1]) + ["value"]
    
    fig = px.histogram(df_snapshot, x="value", nbins=20, color="asset",
                       title=f"Distribución cross-sectional {block} (última fecha)")
    fig.show()

# ============================================================
# 6. Correlación rápida entre horizontes y bloques MSI
# ============================================================
msi_cols = [c for c in momentum_final.columns.get_level_values("feature") if c.startswith("MSI_")]
if msi_cols:
    df_msi = momentum_final.loc[:, pd.IndexSlice[msi_cols, :]]
    # Flatten multiindex: feature + asset → single name
    df_msi_flat = df_msi.copy()
    df_msi_flat.columns = [f"{feat}_{asset}" for feat, asset in df_msi_flat.columns]
    
    corr = df_msi_flat.corr()
    fig = px.imshow(corr, text_auto=False, color_continuous_scale="RdBu_r",
                    title="Correlación entre features MSI")
    fig.show()

# ============================================================
# 7. Chequeo de fechas y NaNs
# ============================================================
print(f"Rango de fechas: {momentum_final.index.min()} → {momentum_final.index.max()}")
print(f"Total NaNs en momentum_final: {momentum_final.isna().sum().sum()}")

df_nan = momentum_final.isna().sum(axis=1).reset_index()
df_nan.columns = list(df_nan.columns[:-1]) + ["nans"]

fig = px.line(df_nan, x=df_nan.columns[0], y="nans", title="NaNs por fecha")
fig.show()

print("SECCIÓN 7 completada: Visualización y sanity checks listos.")

# 8. Exports

In [ ]:
import os
import pandas as pd

# ========================================
# CONFIG: robust project-root-aware paths
# ========================================

# 1. Detectar automáticamente la raíz del proyecto
#    Buscamos la carpeta que contiene 'data', asumimos que es la raíz
def find_project_root(marker_folder="data"):
    current_path = os.path.abspath(os.getcwd())
    while True:
        if marker_folder in os.listdir(current_path):
            return current_path
        parent = os.path.dirname(current_path)
        if parent == current_path:
            raise RuntimeError(f"No se encontró la carpeta '{marker_folder}' en la jerarquía de directorios.")
        current_path = parent

project_root = find_project_root("data")
print("Project root detected:", project_root)

# 2. Carpeta base para features de assets
base_feature_dir = os.path.join(project_root, "data", "features", "asset")
os.makedirs(base_feature_dir, exist_ok=True)

# ========================================
# EXPORTAR momentum por asset
# ========================================

# momentum_final es el DataFrame MultiIndex (feature x asset)
for asset in momentum_final.columns.get_level_values("asset").unique():
    asset_dir = os.path.join(base_feature_dir, asset)
    os.makedirs(asset_dir, exist_ok=True)
    
    # Filtrar columnas del asset
    df_asset = momentum_final.xs(asset, level="asset", axis=1)
    
    # Guardar parquet
    filename = os.path.join(asset_dir, "momentum.parquet")
    df_asset.to_parquet(filename)
    print(f"Momentum exported for {asset} -> {filename}")